# Setup

In [1]:
import os
import sys
parent_dir = os.path.abspath(os.path.join(os.path.dirname("inpire.py"), '..'))
sys.path.append(parent_dir)

In [2]:
from inspire import InspireClassifier

import logging
import warnings
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import BorderlineSMOTE
from sklearn.preprocessing import StandardScaler
from ucimlrepo import fetch_ucirepo 
import kagglehub
from kagglehub import KaggleDatasetAdapter
from tqdm.notebook import tqdm
import json

np.random.seed(42)
logging.basicConfig(level=logging.INFO)
warnings.simplefilter("ignore")

In [3]:
def generate_training_raport(model, X_test, y_test):
    y_pred = model.predict(X_test, level=logging.CRITICAL)
    print(classification_report(y_test, y_pred), "\n")

    n = int(len(model.history[3:]) / 2)
    print("_________Training report_________\n")
    print(f"Removed noise: {sum(model.history[0]['removed_mask'])}")
    if "border_mask" in model.history[1]:
        oversampling = True
        print(f"Border samples: {sum(model.history[1]['border_mask'])}")
    else:
        oversampling = False

    if "folds" in model.history[2]:
        print(f"Number of folds: {len(model.history[2]['folds'])}")

    print()

    for i in range(n):
        print(f"Batch {i+1}")
        if i > 0:
            changed_preds = sum(
                model.history[2 * i + 3]["preds"]
                != model.history[2 * (i - 1) + 3]["preds"]
            )
            print(f"Changed predictions: {changed_preds}")
        if oversampling:
            print(f"Wrong predictions: {sum(model.history[2*i+4]['misclass_mask'])}")
            print(
                f"Low confidence in predictions: {sum(model.history[2*i+4]['condidence_mask'])}"
            )
            print(f"Val BP samples: {sum(model.history[2*i+4]['val_bp_mask'])}")
            print(f"BP samples: {sum(model.history[2*i+4]['bp_mask'])}")
            print(
                f"Oversampling indces: {len(model.history[2*i+4]['oversampling_indeces'])}"
            )
            print(f"X_synth size: {len(model.history[2*i+4]['X_synth'])}")
        print()

    changed_preds = sum(
        model.history[3]["preds"] != model.history[2 * (n - 1) + 3]["preds"]
    )
    print(f"Changed preds (1 -> {n} batch): {changed_preds}")

# Polish companies bankruptcy

## Data

In [4]:
polish_companies_bankruptcy = fetch_ucirepo(id=365) 
  
X = polish_companies_bankruptcy.data.features
y = polish_companies_bankruptcy.data.targets

In [5]:
df = pd.DataFrame(X)
df["target"] = y

In [6]:
df.dropna(inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19967 entries, 0 to 43400
Data columns (total 66 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   year    19967 non-null  int64  
 1   A1      19967 non-null  float64
 2   A2      19967 non-null  float64
 3   A3      19967 non-null  float64
 4   A4      19967 non-null  float64
 5   A5      19967 non-null  float64
 6   A6      19967 non-null  float64
 7   A7      19967 non-null  float64
 8   A8      19967 non-null  float64
 9   A9      19967 non-null  float64
 10  A10     19967 non-null  float64
 11  A11     19967 non-null  float64
 12  A12     19967 non-null  float64
 13  A13     19967 non-null  float64
 14  A14     19967 non-null  float64
 15  A15     19967 non-null  float64
 16  A16     19967 non-null  float64
 17  A17     19967 non-null  float64
 18  A18     19967 non-null  float64
 19  A19     19967 non-null  float64
 20  A20     19967 non-null  float64
 21  A21     19967 non-null  float64
 22  A22

In [7]:
y = df["target"].to_numpy()
X = df.drop("target", axis=1).to_numpy()

In [8]:
scaler_1 = StandardScaler() 
X = scaler_1.fit_transform(X)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, shuffle=True
)
X_test, X_val, y_test, y_val = train_test_split(
    X_test, y_test, test_size=0.5, random_state=42, shuffle=True
)

In [10]:
print("train disbalance:", Counter(y_train))
print("test disbalance:", Counter(y_test))
print("val disbalance:", Counter(y_val))

train disbalance: Counter({np.int64(0): 14648, np.int64(1): 327})
test disbalance: Counter({np.int64(0): 2436, np.int64(1): 60})
val disbalance: Counter({np.int64(0): 2451, np.int64(1): 45})


In [11]:
minority_class = 1

## INSPIRE

### All features

In [24]:
model = InspireClassifier(
    n_estimators=50,
    base_estimator=DecisionTreeClassifier(
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    ),
)

model.fit(
    X_train,
    y_train,
    X_val,
    y_val,
    save_history_=True,
    oversampling_ratio=0.5,
    cleanup_=False,
    level=logging.INFO,
    bp_kwargs={"neighbours": 2},
    br_kwargs={"br_treshold": 0.7}
)

INFO:inspire:Fitting KNN index... (1.01 seconds)
INFO:inspire:Removing outliers... (0.00 seconds)
INFO:inspire:Identified minority class: 1
INFO:inspire:Identified number of classes: 2
INFO:inspire:Fitting KNN index for validation set... (0.03 seconds)
INFO:inspire:Identifying border regions... (0.02 seconds)
INFO:inspire:Training models (step 0)... (0.71 seconds)
INFO:inspire:Performing adaptive optimizations (step 1)... (0.01 seconds)
INFO:inspire:Training models (step 1)... (1.20 seconds)
INFO:inspire:Performing adaptive optimizations (step 2)... (0.01 seconds)
INFO:inspire:Training models (step 2)... (1.25 seconds)
INFO:inspire:Performing adaptive optimizations (step 3)... (0.01 seconds)
INFO:inspire:Training models (step 3)... (1.25 seconds)
INFO:inspire:Performing adaptive optimizations (step 4)... (0.01 seconds)
INFO:inspire:Training models (step 4)... (1.19 seconds)
INFO:inspire:Performing adaptive optimizations (step 5)... (0.01 seconds)
INFO:inspire:Training models (step 5)..

InspireClassifier(base_estimator=DecisionTreeClassifier(max_depth=20,
                                                        max_features=0.8,
                                                        min_samples_leaf=3,
                                                        min_samples_split=5,
                                                        random_state=42),
                  n_estimators=50)

In [26]:
generate_training_raport(model, X_test, y_test)

              precision    recall  f1-score   support

           0       0.99      0.97      0.98      2436
           1       0.30      0.45      0.36        60

    accuracy                           0.96      2496
   macro avg       0.64      0.71      0.67      2496
weighted avg       0.97      0.96      0.97      2496
 

_________Training report_________

Removed noise: 0
Border samples: 327
Number of folds: 50

Batch 1
Wrong predictions: 47
Low confidence in predictions: 64
Val BP samples: 97
BP samples: 141
Oversampling indces: 141
X_synth size: 7160

Batch 2
Changed predictions: 16
Wrong predictions: 49
Low confidence in predictions: 105
Val BP samples: 143
BP samples: 174
Oversampling indces: 174
X_synth size: 7160

Batch 3
Changed predictions: 26
Wrong predictions: 59
Low confidence in predictions: 297
Val BP samples: 317
BP samples: 240
Oversampling indces: 240
X_synth size: 7160

Batch 4
Changed predictions: 24
Wrong predictions: 67
Low confidence in predictions: 204
Val B

In [27]:
model = InspireClassifier(
    n_estimators=50,
    base_estimator=DecisionTreeClassifier(
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    ),
)

model.fit(
    X_train,
    y_train,
    X_val,
    y_val,
    save_history_=True,
    oversampling_ratio=1,
    cleanup_=False,
    level=logging.INFO,
    bp_kwargs={"neighbours": 2},
    br_kwargs={"br_treshold": 0.7}
)

INFO:inspire:Fitting KNN index... (0.95 seconds)
INFO:inspire:Removing outliers... (0.00 seconds)
INFO:inspire:Identified minority class: 1
INFO:inspire:Identified number of classes: 2
INFO:inspire:Fitting KNN index for validation set... (0.04 seconds)
INFO:inspire:Identifying border regions... (0.02 seconds)
INFO:inspire:Training models (step 0)... (0.72 seconds)
INFO:inspire:Performing adaptive optimizations (step 1)... (0.01 seconds)
INFO:inspire:Training models (step 1)... (1.91 seconds)
INFO:inspire:Performing adaptive optimizations (step 2)... (0.02 seconds)
INFO:inspire:Training models (step 2)... (1.76 seconds)
INFO:inspire:Performing adaptive optimizations (step 3)... (0.01 seconds)
INFO:inspire:Training models (step 3)... (1.99 seconds)
INFO:inspire:Performing adaptive optimizations (step 4)... (0.01 seconds)
INFO:inspire:Training models (step 4)... (1.82 seconds)
INFO:inspire:Performing adaptive optimizations (step 5)... (0.01 seconds)
INFO:inspire:Training models (step 5)..

InspireClassifier(base_estimator=DecisionTreeClassifier(max_depth=20,
                                                        max_features=0.8,
                                                        min_samples_leaf=3,
                                                        min_samples_split=5,
                                                        random_state=42),
                  n_estimators=50)

In [31]:
generate_training_raport(model, X_test, y_test)

              precision    recall  f1-score   support

           0       0.99      0.93      0.96      2436
           1       0.17      0.60      0.27        60

    accuracy                           0.92      2496
   macro avg       0.58      0.76      0.61      2496
weighted avg       0.97      0.92      0.94      2496
 

_________Training report_________

Removed noise: 0
Border samples: 327
Number of folds: 50

Batch 1
Wrong predictions: 47
Low confidence in predictions: 64
Val BP samples: 97
BP samples: 141
Oversampling indces: 141
X_synth size: 14321

Batch 2
Changed predictions: 19
Wrong predictions: 52
Low confidence in predictions: 185
Val BP samples: 225
BP samples: 217
Oversampling indces: 217
X_synth size: 14321

Batch 3
Changed predictions: 65
Wrong predictions: 103
Low confidence in predictions: 391
Val BP samples: 413
BP samples: 258
Oversampling indces: 258
X_synth size: 14321

Batch 4
Changed predictions: 50
Wrong predictions: 131
Low confidence in predictions: 292


In [ ]:
recalls = []
precisions = []
fscores = []

for i in tqdm(range(10)):
    model_i = InspireClassifier(
        n_estimators=50,
        base_estimator=DecisionTreeClassifier(
            max_depth=20,
            min_samples_split=5,
            min_samples_leaf=3,
            max_features=0.8,
            random_state=42,
        ),
    )

    model_i.fit(
        X_train,
        y_train,
        X_val,
        y_val,
        save_history_=False,
        oversampling_per_step=1000,
        cleanup_=True,
        level=logging.CRITICAL,
        bp_kwargs={"neighbours": 2},
        br_kwargs={"br_treshold": 0.7},
    )

    y_pred = model_i.predict(
        X_test,
        level=logging.CRITICAL,
    )
    report = classification_report(y_test, y_pred, output_dict=True)

    recalls.append(report[f"{minority_class}"]["recall"])
    precisions.append(report[f"{minority_class}"]["precision"])
    fscores.append(report[f"{minority_class}"]["f1-score"])

  0%|          | 0/10 [00:00<?, ?it/s]

In [14]:
polish_companies_results = {}
polish_companies_results["all"] = {
    "recalls": recalls,
    "precisions": precisions,
    "fscores": fscores,
}

In [15]:
del model
del model_i

### No bagging

In [12]:
model = InspireClassifier(
    n_estimators=1,
    base_estimator=DecisionTreeClassifier(
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    ),
)

model.fit(
    X_train,
    y_train,
    X_val,
    y_val,
    step_size=1,
    save_history_=True,
    oversampling_per_step=1000,
    bagging_=False,
    cleanup_=False,
    level=logging.INFO,
    bp_kwargs={"neighbours": 2},
    br_kwargs={"br_treshold": 0.7}
)

INFO:inspire:Fitting KNN index... (0.91 seconds)
INFO:inspire:Removing outliers... (0.00 seconds)
INFO:inspire:Identified minority class: 1
INFO:inspire:Identified number of classes: 2
INFO:inspire:Fitting KNN index for validation set... (0.04 seconds)
INFO:inspire:Identifying border regions... (0.02 seconds)


InspireClassifier(base_estimator=DecisionTreeClassifier(max_depth=10,
                                                        max_features=0.8,
                                                        min_samples_leaf=3,
                                                        min_samples_split=5,
                                                        random_state=42),
                  n_estimators=1)

In [13]:
generate_training_raport(model, X_test, y_test)

              precision    recall  f1-score   support

           0       0.98      0.99      0.99      2436
           1       0.39      0.32      0.35        60

    accuracy                           0.97      2496
   macro avg       0.69      0.65      0.67      2496
weighted avg       0.97      0.97      0.97      2496
 

_________Training report_________

Removed noise: 0
Border samples: 327
Number of folds: 8

Batch 1
Wrong predictions: 61
Low confidence in predictions: 0
Val BP samples: 61
BP samples: 92
Oversampling indces: 92
X_synth size: 1000

Changed preds (1 -> 1 batch): 0


In [ ]:
recalls = []
precisions = []
fscores = []

for i in tqdm(range(10)):
    model_i = InspireClassifier(
        n_estimators=10,
        base_estimator=DecisionTreeClassifier(
            max_depth=10,
            min_samples_split=5,
            min_samples_leaf=3,
            max_features=0.8,
            random_state=42,
        ),
    )

    model_i.fit(
        X_train,
        y_train,
        X_val,
        y_val,
        save_history_=False,
        oversampling_per_step=1000,
        cleanup_=True,
        level=logging.CRITICAL,
        bp_kwargs={"neighbours": 2},
        br_kwargs={"br_treshold": 0.7},
    )

    y_pred = model_i.predict(
        X_test,
        level=logging.CRITICAL,
    )
    report = classification_report(y_test, y_pred, output_dict=True)

    recalls.append(report[f"{minority_class}"]["recall"])
    precisions.append(report[f"{minority_class}"]["precision"])
    fscores.append(report[f"{minority_class}"]["f1-score"])

  0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
polish_companies_results["no_bagging"] = {
    "recalls": recalls,
    "precisions": precisions,
    "fscores": fscores,
}

In [ ]:
del model
del model_i

### No SMOTE

In [21]:
model = InspireClassifier(
    n_estimators=100,
    base_estimator=DecisionTreeClassifier(
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    ),
)

model.fit(
    X_train,
    y_train,
    X_val,
    y_val,
    save_history_=True,
    perform_oversampling_=False,
    oversampling_per_step=1000,
    cleanup_=False,
    level=logging.INFO,
    bp_kwargs={"neighbours": 2},
    br_kwargs={"br_treshold": 0.7}
)

INFO:inspire:Fitting KNN index... (0.64 seconds)
INFO:inspire:Removing outliers... (0.00 seconds)
INFO:inspire:Identified minority class: 1
INFO:inspire:Identified number of classes: 2
INFO:inspire:Fitting KNN index for validation set... (0.03 seconds)
INFO:inspire:Training models (step 0)... (0.54 seconds)
INFO:inspire:Performing adaptive optimizations (step 1)... (0.00 seconds)
INFO:inspire:Training models (step 1)... (0.52 seconds)
INFO:inspire:Performing adaptive optimizations (step 2)... (0.00 seconds)
INFO:inspire:Training models (step 2)... (0.63 seconds)
INFO:inspire:Performing adaptive optimizations (step 3)... (0.00 seconds)
INFO:inspire:Training models (step 3)... (0.68 seconds)
INFO:inspire:Performing adaptive optimizations (step 4)... (0.00 seconds)
INFO:inspire:Training models (step 4)... (0.60 seconds)
INFO:inspire:Performing adaptive optimizations (step 5)... (0.00 seconds)
INFO:inspire:Training models (step 5)... (0.63 seconds)
INFO:inspire:Performing adaptive optimiza

InspireClassifier(base_estimator=DecisionTreeClassifier(max_depth=10,
                                                        max_features=0.8,
                                                        min_samples_leaf=3,
                                                        min_samples_split=5,
                                                        random_state=42),
                  n_estimators=100)

In [22]:
generate_training_raport(model, X_test, y_test)

              precision    recall  f1-score   support

           0       0.98      0.99      0.99      2436
           1       0.48      0.27      0.34        60

    accuracy                           0.98      2496
   macro avg       0.73      0.63      0.67      2496
weighted avg       0.97      0.98      0.97      2496
 

_________Training report_________

Removed noise: 0

Batch 1

Batch 2
Changed predictions: 9

Batch 3
Changed predictions: 4

Batch 4
Changed predictions: 5

Batch 5
Changed predictions: 4

Batch 6
Changed predictions: 2

Batch 7
Changed predictions: 1

Batch 8
Changed predictions: 1

Batch 9
Changed predictions: 3

Batch 10
Changed predictions: 3

Batch 11
Changed predictions: 0

Batch 12
Changed predictions: 0

Batch 13
Changed predictions: 0

Batch 14
Changed predictions: 0

Batch 15
Changed predictions: 1

Batch 16
Changed predictions: 1

Batch 17
Changed predictions: 3

Batch 18
Changed predictions: 1

Batch 19
Changed predictions: 1

Batch 20
Changed predic

In [23]:
recalls = []
precisions = []
fscores = []

for i in tqdm(range(10)):
    model_i = InspireClassifier(
        n_estimators=100,
        base_estimator=DecisionTreeClassifier(
            max_depth=10,
            min_samples_split=5,
            min_samples_leaf=3,
            max_features=0.8,
            random_state=42,
        ),
    )

    model_i.fit(
        X_train,
        y_train,
        X_val,
        y_val,
        save_history_=False,
        oversampling_per_step=1000,
        cleanup_=True,
        level=logging.CRITICAL,
        bp_kwargs={"neighbours": 2},
        br_kwargs={"br_treshold": 0.7},
    )

    y_pred = model_i.predict(
        X_test,
        level=logging.CRITICAL,
    )
    report = classification_report(y_test, y_pred, output_dict=True)

    recalls.append(report[f"{minority_class}"]["recall"])
    precisions.append(report[f"{minority_class}"]["precision"])
    fscores.append(report[f"{minority_class}"]["f1-score"])

  0%|          | 0/10 [00:00<?, ?it/s]

In [24]:
polish_companies_results["no_smote"] = {
    "recalls": recalls,
    "precisions": precisions,
    "fscores": fscores,
}

In [25]:
del model
del model_i

## RFC + BordelineSMOTE

In [26]:
rfc = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=3,
    max_features=0.8,
    random_state=42,
)

In [27]:
smote = BorderlineSMOTE(kind='borderline-1', random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

In [28]:
rfc.fit(X_resampled, y_resampled)

RandomForestClassifier(max_depth=10, max_features=0.8, min_samples_leaf=3,
                       min_samples_split=5, random_state=42)

In [29]:
y_pred_rfc = rfc.predict(X_test)

print(classification_report(y_test, y_pred_rfc))

              precision    recall  f1-score   support

           0       0.99      0.90      0.95      2436
           1       0.14      0.63      0.23        60

    accuracy                           0.90      2496
   macro avg       0.56      0.77      0.59      2496
weighted avg       0.97      0.90      0.93      2496



In [30]:
recalls = []
precisions = []
fscores = []
for i in tqdm(range(10)):
    rfc_i = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    )
    rfc_i.fit(X_resampled, y_resampled)
    
    y_pred = rfc_i.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)

    recalls.append(report[f"{minority_class}"]["recall"])
    precisions.append(report[f"{minority_class}"]["precision"])
    fscores.append(report[f"{minority_class}"]["f1-score"])

  0%|          | 0/10 [00:00<?, ?it/s]

In [31]:
polish_companies_results["rfc_smote"] = {
    "recalls": recalls,
    "precisions": precisions,
    "fscores": fscores,
}

In [32]:
del rfc
del rfc_i

### RFC

In [28]:
rfc = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=3,
    max_features=0.8,
    random_state=42,
)

In [29]:
rfc.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, max_features=0.8, min_samples_leaf=3,
                       min_samples_split=5, random_state=42)

In [30]:
y_pred_rfc = rfc.predict(X_test)

print(classification_report(y_test, y_pred_rfc))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99      2436
           1       0.90      0.15      0.26        60

    accuracy                           0.98      2496
   macro avg       0.94      0.57      0.62      2496
weighted avg       0.98      0.98      0.97      2496



In [36]:
recalls = []
precisions = []
fscores = []
for i in tqdm(range(10)):
    rfc_i = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    )
    rfc_i.fit(X_test, y_test)
    
    y_pred = rfc_i.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)

    recalls.append(report[f"{minority_class}"]["recall"])
    precisions.append(report[f"{minority_class}"]["precision"])
    fscores.append(report[f"{minority_class}"]["f1-score"])

  0%|          | 0/10 [00:00<?, ?it/s]

In [37]:
polish_companies_results["rfc"] = {
    "recalls": recalls,
    "precisions": precisions,
    "fscores": fscores,
}

In [38]:
del rfc
del rfc_i

In [39]:
with open("results/polish_companies_results.json", "w") as f:
    json.dump(polish_companies_results, f, indent=4)

In [40]:
del polish_companies_results

# Breast Cancer Wisconsin

## Data

In [41]:
breast_cancer_wisconsin_diagnostic = fetch_ucirepo(id=17) 
  
X = breast_cancer_wisconsin_diagnostic.data.features 
y = breast_cancer_wisconsin_diagnostic.data.targets

In [42]:
df = pd.DataFrame(X)
df["target"] = y

In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   radius1             569 non-null    float64
 1   texture1            569 non-null    float64
 2   perimeter1          569 non-null    float64
 3   area1               569 non-null    float64
 4   smoothness1         569 non-null    float64
 5   compactness1        569 non-null    float64
 6   concavity1          569 non-null    float64
 7   concave_points1     569 non-null    float64
 8   symmetry1           569 non-null    float64
 9   fractal_dimension1  569 non-null    float64
 10  radius2             569 non-null    float64
 11  texture2            569 non-null    float64
 12  perimeter2          569 non-null    float64
 13  area2               569 non-null    float64
 14  smoothness2         569 non-null    float64
 15  compactness2        569 non-null    float64
 16  concavit

In [44]:
X = df.drop("target", axis=1).to_numpy()
y = df["target"].to_numpy()

In [45]:
Counter(y)

Counter({'B': 357, 'M': 212})

In [46]:
y = (y == 'M').astype(int)

In [47]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, shuffle=True
)
X_test, X_val, y_test, y_val = train_test_split(
    X_test, y_test, test_size=0.5, random_state=42, shuffle=True
)

In [48]:
print("train disbalance:", Counter(y_train))
print("test disbalance:", Counter(y_test))
print("val disbalance:", Counter(y_val))

train disbalance: Counter({np.int64(0): 268, np.int64(1): 158})
test disbalance: Counter({np.int64(0): 42, np.int64(1): 29})
val disbalance: Counter({np.int64(0): 47, np.int64(1): 25})


In [49]:
minority_class = 1

## INSPIRE

### All features

In [50]:
model = InspireClassifier(
    n_estimators=100,
    base_estimator=DecisionTreeClassifier(
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    ),
)

model.fit(
    X_train,
    y_train,
    X_val,
    y_val,
    save_history_=True,
    oversampling_ratio=1,
    cleanup_=False,
    level=logging.INFO,
    bp_kwargs={"neighbours": 2},
    br_kwargs={"br_treshold": 0.7}
)

INFO:inspire:Fitting KNN index... (0.06 seconds)
INFO:inspire:Removing outliers... (0.00 seconds)
INFO:inspire:Identified minority class: 1
INFO:inspire:Identified number of classes: 2


INFO:inspire:Fitting KNN index for validation set... (0.01 seconds)
INFO:inspire:Identifying border regions... (0.00 seconds)
INFO:inspire:Training models (step 0)... (0.01 seconds)
INFO:inspire:Performing adaptive optimizations (step 1)... (0.00 seconds)
INFO:inspire:Training models (step 1)... (0.01 seconds)
INFO:inspire:Performing adaptive optimizations (step 2)... (0.00 seconds)
INFO:inspire:Training models (step 2)... (0.01 seconds)
INFO:inspire:Performing adaptive optimizations (step 3)... (0.00 seconds)
INFO:inspire:Training models (step 3)... (0.01 seconds)
INFO:inspire:Performing adaptive optimizations (step 4)... (0.00 seconds)
INFO:inspire:Training models (step 4)... (0.01 seconds)
INFO:inspire:Performing adaptive optimizations (step 5)... (0.00 seconds)
INFO:inspire:Training models (step 5)... (0.01 seconds)
INFO:inspire:Performing adaptive optimizations (step 6)... (0.00 seconds)
INFO:inspire:Training models (step 6)... (0.01 seconds)
INFO:inspire:Performing adaptive optim

InspireClassifier(base_estimator=DecisionTreeClassifier(max_depth=10,
                                                        max_features=0.8,
                                                        min_samples_leaf=3,
                                                        min_samples_split=5,
                                                        random_state=42),
                  n_estimators=100)

In [51]:
generate_training_raport(model, X_test, y_test)

              precision    recall  f1-score   support

           0       1.00      0.90      0.95        42
           1       0.88      1.00      0.94        29

    accuracy                           0.94        71
   macro avg       0.94      0.95      0.94        71
weighted avg       0.95      0.94      0.94        71
 

_________Training report_________

Removed noise: 0
Border samples: 40
Number of folds: 100

Batch 1
Wrong predictions: 3
Low confidence in predictions: 0
Val BP samples: 3
BP samples: 6
Oversampling indces: 4
X_synth size: 110

Batch 2
Changed predictions: 0
Wrong predictions: 3
Low confidence in predictions: 4
Val BP samples: 5
BP samples: 9
Oversampling indces: 7
X_synth size: 110

Batch 3
Changed predictions: 1
Wrong predictions: 2
Low confidence in predictions: 4
Val BP samples: 5
BP samples: 9
Oversampling indces: 7
X_synth size: 110

Batch 4
Changed predictions: 1
Wrong predictions: 1
Low confidence in predictions: 5
Val BP samples: 6
BP samples: 9
Oversam

In [52]:
recalls = []
precisions = []
fscores = []

for i in tqdm(range(30)):
    model_i = InspireClassifier(
        n_estimators=100,
        base_estimator=DecisionTreeClassifier(
            max_depth=10,
            min_samples_split=5,
            min_samples_leaf=3,
            max_features=0.8,
            random_state=42,
        ),
    )

    model_i.fit(
        X_train,
        y_train,
        X_val,
        y_val,
        save_history_=False,
        oversampling_ratio=1,
        cleanup_=True,
        level=logging.CRITICAL,
        bp_kwargs={"neighbours": 2},
        br_kwargs={"br_treshold": 0.7},
    )

    y_pred = model_i.predict(
        X_test,
        level=logging.CRITICAL,
    )
    report = classification_report(y_test, y_pred, output_dict=True)

    recalls.append(report[f"{minority_class}"]["recall"])
    precisions.append(report[f"{minority_class}"]["precision"])
    fscores.append(report[f"{minority_class}"]["f1-score"])

  0%|          | 0/30 [00:00<?, ?it/s]

In [53]:
wisconsin_results = {}
wisconsin_results["all"] = {
    "recalls": recalls,
    "precisions": precisions,
    "fscores": fscores,
}
del model
del model_i

### No bagging

In [54]:
model = InspireClassifier(
    n_estimators=100,
    base_estimator=DecisionTreeClassifier(
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    ),
)

model.fit(
    X_train,
    y_train,
    X_val,
    y_val,
    save_history_=True,
    oversampling_ratio=1,
    bagging_=False,
    cleanup_=False,
    level=logging.INFO,
    bp_kwargs={"neighbours": 2},
    br_kwargs={"br_treshold": 0.7}
)

INFO:inspire:Fitting KNN index... (0.06 seconds)
INFO:inspire:Removing outliers... (0.00 seconds)
INFO:inspire:Identified minority class: 1
INFO:inspire:Identified number of classes: 2


INFO:inspire:Fitting KNN index for validation set... (0.01 seconds)
INFO:inspire:Identifying border regions... (0.00 seconds)
INFO:inspire:Training models (step 0)... (0.08 seconds)
INFO:inspire:Performing adaptive optimizations (step 1)... (0.00 seconds)
INFO:inspire:Training models (step 1)... (0.11 seconds)
INFO:inspire:Performing adaptive optimizations (step 2)... (0.00 seconds)
INFO:inspire:Training models (step 2)... (0.10 seconds)
INFO:inspire:Performing adaptive optimizations (step 3)... (0.00 seconds)
INFO:inspire:Training models (step 3)... (0.10 seconds)
INFO:inspire:Performing adaptive optimizations (step 4)... (0.00 seconds)
INFO:inspire:Training models (step 4)... (0.10 seconds)
INFO:inspire:Performing adaptive optimizations (step 5)... (0.00 seconds)
INFO:inspire:Training models (step 5)... (0.10 seconds)
INFO:inspire:Performing adaptive optimizations (step 6)... (0.00 seconds)
INFO:inspire:Training models (step 6)... (0.10 seconds)
INFO:inspire:Performing adaptive optim

InspireClassifier(base_estimator=DecisionTreeClassifier(max_depth=10,
                                                        max_features=0.8,
                                                        min_samples_leaf=3,
                                                        min_samples_split=5,
                                                        random_state=42),
                  n_estimators=100)

In [55]:
generate_training_raport(model, X_test, y_test)

              precision    recall  f1-score   support

           0       0.97      0.88      0.93        42
           1       0.85      0.97      0.90        29

    accuracy                           0.92        71
   macro avg       0.91      0.92      0.91        71
weighted avg       0.92      0.92      0.92        71
 

_________Training report_________

Removed noise: 0
Border samples: 40
Number of folds: 8

Batch 1
Wrong predictions: 3
Low confidence in predictions: 0
Val BP samples: 3
BP samples: 5
Oversampling indces: 5
X_synth size: 110

Batch 2
Changed predictions: 0
Wrong predictions: 3
Low confidence in predictions: 0
Val BP samples: 3
BP samples: 5
Oversampling indces: 5
X_synth size: 110

Batch 3
Changed predictions: 0
Wrong predictions: 3
Low confidence in predictions: 1
Val BP samples: 4
BP samples: 6
Oversampling indces: 6
X_synth size: 110

Batch 4
Changed predictions: 0
Wrong predictions: 3
Low confidence in predictions: 0
Val BP samples: 3
BP samples: 5
Oversampl

In [56]:
recalls = []
precisions = []
fscores = []

for i in tqdm(range(30)):
    model_i = InspireClassifier(
        n_estimators=100,
        base_estimator=DecisionTreeClassifier(
            max_depth=10,
            min_samples_split=5,
            min_samples_leaf=3,
            max_features=0.8,
            random_state=42,
        ),
    )

    model_i.fit(
        X_train,
        y_train,
        X_val,
        y_val,
        save_history_=False,
        oversampling_ratio=1,
        bagging_=False,
        cleanup_=True,
        level=logging.CRITICAL,
        bp_kwargs={"neighbours": 2},
        br_kwargs={"br_treshold": 0.7},
    )

    y_pred = model_i.predict(
        X_test,
        level=logging.CRITICAL,
    )
    report = classification_report(y_test, y_pred, output_dict=True)

    recalls.append(report[f"{minority_class}"]["recall"])
    precisions.append(report[f"{minority_class}"]["precision"])
    fscores.append(report[f"{minority_class}"]["f1-score"])

  0%|          | 0/30 [00:00<?, ?it/s]

In [57]:
wisconsin_results["no_bagging"] = {
    "recalls": recalls,
    "precisions": precisions,
    "fscores": fscores,
}
del model
del model_i

### No SMOTE

In [58]:
model = InspireClassifier(
    n_estimators=100,
    base_estimator=DecisionTreeClassifier(
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    ),
)

model.fit(
    X_train,
    y_train,
    X_val,
    y_val,
    save_history_=True,
    oversampling_ratio=1,
    perform_oversampling_=False,
    cleanup_=False,
    level=logging.INFO,
    bp_kwargs={"neighbours": 2},
    br_kwargs={"br_treshold": 0.7}
)

INFO:inspire:Fitting KNN index... (0.08 seconds)
INFO:inspire:Removing outliers... (0.00 seconds)
INFO:inspire:Identified minority class: 1
INFO:inspire:Identified number of classes: 2
INFO:inspire:Fitting KNN index for validation set... (0.00 seconds)
INFO:inspire:Training models (step 0)... (0.01 seconds)
INFO:inspire:Performing adaptive optimizations (step 1)... (0.00 seconds)
INFO:inspire:Training models (step 1)... (0.01 seconds)
INFO:inspire:Performing adaptive optimizations (step 2)... (0.00 seconds)
INFO:inspire:Training models (step 2)... (0.00 seconds)
INFO:inspire:Performing adaptive optimizations (step 3)... (0.00 seconds)
INFO:inspire:Training models (step 3)... (0.00 seconds)
INFO:inspire:Performing adaptive optimizations (step 4)... (0.00 seconds)
INFO:inspire:Training models (step 4)... (0.00 seconds)
INFO:inspire:Performing adaptive optimizations (step 5)... (0.00 seconds)
INFO:inspire:Training models (step 5)... (0.01 seconds)
INFO:inspire:Performing adaptive optimiza

InspireClassifier(base_estimator=DecisionTreeClassifier(max_depth=10,
                                                        max_features=0.8,
                                                        min_samples_leaf=3,
                                                        min_samples_split=5,
                                                        random_state=42),
                  n_estimators=100)

In [59]:
generate_training_raport(model, X_test, y_test)

              precision    recall  f1-score   support

           0       0.97      0.93      0.95        42
           1       0.90      0.97      0.93        29

    accuracy                           0.94        71
   macro avg       0.94      0.95      0.94        71
weighted avg       0.95      0.94      0.94        71
 

_________Training report_________

Removed noise: 0

Batch 1

Batch 2
Changed predictions: 0

Batch 3
Changed predictions: 0

Batch 4
Changed predictions: 0

Batch 5
Changed predictions: 0

Batch 6
Changed predictions: 0

Batch 7
Changed predictions: 0

Batch 8
Changed predictions: 0

Batch 9
Changed predictions: 0

Batch 10
Changed predictions: 0

Batch 11
Changed predictions: 0

Batch 12
Changed predictions: 0

Batch 13
Changed predictions: 0

Batch 14
Changed predictions: 0

Batch 15
Changed predictions: 0

Batch 16
Changed predictions: 0

Batch 17
Changed predictions: 0

Batch 18
Changed predictions: 0

Batch 19
Changed predictions: 0

Batch 20
Changed predic

In [60]:
recalls = []
precisions = []
fscores = []

for i in tqdm(range(30)):
    model_i = InspireClassifier(
        n_estimators=100,
        base_estimator=DecisionTreeClassifier(
            max_depth=10,
            min_samples_split=5,
            min_samples_leaf=3,
            max_features=0.8,
            random_state=42,
        ),
    )

    model_i.fit(
        X_train,
        y_train,
        X_val,
        y_val,
        save_history_=False,
        oversampling_ratio=1,
        perform_oversampling_=False,
        cleanup_=True,
        level=logging.CRITICAL,
        bp_kwargs={"neighbours": 2},
        br_kwargs={"br_treshold": 0.7},
    )

    y_pred = model_i.predict(
        X_test,
        level=logging.CRITICAL,
    )
    report = classification_report(y_test, y_pred, output_dict=True)

    recalls.append(report[f"{minority_class}"]["recall"])
    precisions.append(report[f"{minority_class}"]["precision"])
    fscores.append(report[f"{minority_class}"]["f1-score"])

  0%|          | 0/30 [00:00<?, ?it/s]

In [61]:
wisconsin_results["no_smote"] = {
    "recalls": recalls,
    "precisions": precisions,
    "fscores": fscores,
}
del model
del model_i

## RFC + BorderlineSMOTE

In [62]:
rfc = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=3,
    max_features=0.8,
    random_state=42,
)

In [63]:
smote = BorderlineSMOTE(kind='borderline-1', random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

In [64]:
rfc.fit(X_resampled, y_resampled)

RandomForestClassifier(max_depth=10, max_features=0.8, min_samples_leaf=3,
                       min_samples_split=5, random_state=42)

In [65]:
y_pred_rfc = rfc.predict(X_test)

print(classification_report(y_test, y_pred_rfc))

              precision    recall  f1-score   support

           0       0.98      0.95      0.96        42
           1       0.93      0.97      0.95        29

    accuracy                           0.96        71
   macro avg       0.95      0.96      0.96        71
weighted avg       0.96      0.96      0.96        71



In [66]:
recalls = []
precisions = []
fscores = []
for i in tqdm(range(30)):
    rfc_i = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    )
    rfc_i.fit(X_resampled, y_resampled)
    
    y_pred = rfc_i.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)

    recalls.append(report[f"{minority_class}"]["recall"])
    precisions.append(report[f"{minority_class}"]["precision"])
    fscores.append(report[f"{minority_class}"]["f1-score"])

  0%|          | 0/30 [00:00<?, ?it/s]

In [67]:
wisconsin_results["rfc_smote"] = {
    "recalls": recalls,
    "precisions": precisions,
    "fscores": fscores,
}

In [68]:
del rfc
del rfc_i

### RFC

In [69]:
rfc = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=3,
    max_features=0.8,
    random_state=42,
)

In [70]:
rfc.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, max_features=0.8, min_samples_leaf=3,
                       min_samples_split=5, random_state=42)

In [71]:
y_pred_rfc = rfc.predict(X_test)

print(classification_report(y_test, y_pred_rfc))

              precision    recall  f1-score   support

           0       0.95      0.95      0.95        42
           1       0.93      0.93      0.93        29

    accuracy                           0.94        71
   macro avg       0.94      0.94      0.94        71
weighted avg       0.94      0.94      0.94        71



In [72]:
recalls = []
precisions = []
fscores = []
for i in tqdm(range(10)):
    rfc_i = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    )
    rfc_i.fit(X_test, y_test)
    
    y_pred = rfc_i.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)

    recalls.append(report[f"{minority_class}"]["recall"])
    precisions.append(report[f"{minority_class}"]["precision"])
    fscores.append(report[f"{minority_class}"]["f1-score"])

  0%|          | 0/10 [00:00<?, ?it/s]

In [73]:
wisconsin_results["rfc"] = {
    "recalls": recalls,
    "precisions": precisions,
    "fscores": fscores,
}

In [74]:
del rfc
del rfc_i

In [75]:
with open("results/wisconsin_cancer.json", "w") as f:
    json.dump(wisconsin_results, f, indent=4)

In [76]:
del wisconsin_results

# Ionosphere

## Data

In [77]:
ionosphere = fetch_ucirepo(id=52) 
  
X = ionosphere.data.features 
y = ionosphere.data.targets 

In [78]:
df = pd.DataFrame(X)
df["target"] = y

In [79]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 351 entries, 0 to 350
Data columns (total 35 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Attribute1   351 non-null    int64  
 1   Attribute2   351 non-null    int64  
 2   Attribute3   351 non-null    float64
 3   Attribute4   351 non-null    float64
 4   Attribute5   351 non-null    float64
 5   Attribute6   351 non-null    float64
 6   Attribute7   351 non-null    float64
 7   Attribute8   351 non-null    float64
 8   Attribute9   351 non-null    float64
 9   Attribute10  351 non-null    float64
 10  Attribute11  351 non-null    float64
 11  Attribute12  351 non-null    float64
 12  Attribute13  351 non-null    float64
 13  Attribute14  351 non-null    float64
 14  Attribute15  351 non-null    float64
 15  Attribute16  351 non-null    float64
 16  Attribute17  351 non-null    float64
 17  Attribute18  351 non-null    float64
 18  Attribute19  351 non-null    float64
 19  Attribut

In [80]:
X = df.drop("target", axis=1).to_numpy()
y = df["target"].to_numpy()

In [81]:
Counter(y)

Counter({'g': 225, 'b': 126})

In [82]:
y = (y == 'b').astype(int)

In [83]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, shuffle=True
)
X_test, X_val, y_test, y_val = train_test_split(
    X_test, y_test, test_size=0.5, random_state=42, shuffle=True
)

In [84]:
print("train disbalance:", Counter(y_train))
print("test disbalance:", Counter(y_test))
print("val disbalance:", Counter(y_val))

train disbalance: Counter({np.int64(0): 169, np.int64(1): 94})
test disbalance: Counter({np.int64(0): 25, np.int64(1): 19})
val disbalance: Counter({np.int64(0): 31, np.int64(1): 13})


In [85]:
minority_class = 1

## INSPIRE

### All features

In [86]:
model = InspireClassifier(
    n_estimators=100,
    base_estimator=DecisionTreeClassifier(
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    ),
)

model.fit(
    X_train,
    y_train,
    X_val,
    y_val,
    save_history_=True,
    oversampling_ratio=1,
    cleanup_=False,
    level=logging.INFO,
    bp_kwargs={"neighbours": 2},
    br_kwargs={"br_treshold": 0.7}
)

INFO:inspire:Fitting KNN index... (0.01 seconds)
INFO:inspire:Removing outliers... (0.00 seconds)
INFO:inspire:Identified minority class: 1
INFO:inspire:Identified number of classes: 2
INFO:inspire:Fitting KNN index for validation set... (0.00 seconds)
INFO:inspire:Identifying border regions... (0.00 seconds)
INFO:inspire:Training models (step 0)... (0.00 seconds)
INFO:inspire:Performing adaptive optimizations (step 1)... (0.00 seconds)
INFO:inspire:Training models (step 1)... (0.02 seconds)
INFO:inspire:Performing adaptive optimizations (step 2)... (0.00 seconds)
INFO:inspire:Training models (step 2)... (0.03 seconds)
INFO:inspire:Performing adaptive optimizations (step 3)... (0.00 seconds)
INFO:inspire:Training models (step 3)... (0.01 seconds)
INFO:inspire:Performing adaptive optimizations (step 4)... (0.00 seconds)
INFO:inspire:Training models (step 4)... (0.00 seconds)
INFO:inspire:Performing adaptive optimizations (step 5)... (0.00 seconds)
INFO:inspire:Training models (step 5)..

InspireClassifier(base_estimator=DecisionTreeClassifier(max_depth=10,
                                                        max_features=0.8,
                                                        min_samples_leaf=3,
                                                        min_samples_split=5,
                                                        random_state=42),
                  n_estimators=100)

In [87]:
generate_training_raport(model, X_test, y_test)

              precision    recall  f1-score   support

           0       0.92      0.96      0.94        25
           1       0.94      0.89      0.92        19

    accuracy                           0.93        44
   macro avg       0.93      0.93      0.93        44
weighted avg       0.93      0.93      0.93        44
 

_________Training report_________

Removed noise: 0
Border samples: 79
Number of folds: 100

Batch 1
Wrong predictions: 7
Low confidence in predictions: 0
Val BP samples: 7
BP samples: 9
Oversampling indces: 9
X_synth size: 75

Batch 2
Changed predictions: 2
Wrong predictions: 5
Low confidence in predictions: 4
Val BP samples: 9
BP samples: 11
Oversampling indces: 11
X_synth size: 75

Batch 3
Changed predictions: 2
Wrong predictions: 7
Low confidence in predictions: 8
Val BP samples: 10
BP samples: 12
Oversampling indces: 12
X_synth size: 75

Batch 4
Changed predictions: 2
Wrong predictions: 5
Low confidence in predictions: 4
Val BP samples: 8
BP samples: 11
Over

In [88]:
recalls = []
precisions = []
fscores = []

for i in tqdm(range(30)):
    model_i = InspireClassifier(
        n_estimators=100,
        base_estimator=DecisionTreeClassifier(
            max_depth=10,
            min_samples_split=5,
            min_samples_leaf=3,
            max_features=0.8,
            random_state=42,
        ),
    )

    model_i.fit(
        X_train,
        y_train,
        X_val,
        y_val,
        save_history_=False,
        oversampling_ratio=1,
        cleanup_=True,
        level=logging.CRITICAL,
        bp_kwargs={"neighbours": 2},
        br_kwargs={"br_treshold": 0.7},
    )

    y_pred = model_i.predict(
        X_test,
        level=logging.CRITICAL,
    )
    report = classification_report(y_test, y_pred, output_dict=True)

    recalls.append(report[f"{minority_class}"]["recall"])
    precisions.append(report[f"{minority_class}"]["precision"])
    fscores.append(report[f"{minority_class}"]["f1-score"])

  0%|          | 0/30 [00:00<?, ?it/s]

In [89]:
ionosphere_results = {}
ionosphere_results["all"] = {
    "recalls": recalls,
    "precisions": precisions,
    "fscores": fscores,
}
del model
del model_i

### No bagging

In [90]:
model = InspireClassifier(
    n_estimators=100,
    base_estimator=DecisionTreeClassifier(
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    ),
)

model.fit(
    X_train,
    y_train,
    X_val,
    y_val,
    save_history_=True,
    oversampling_ratio=1,
    bagging_=False,
    cleanup_=False,
    level=logging.INFO,
    bp_kwargs={"neighbours": 2},
    br_kwargs={"br_treshold": 0.7}
)

INFO:inspire:Fitting KNN index... (0.01 seconds)
INFO:inspire:Removing outliers... (0.00 seconds)
INFO:inspire:Identified minority class: 1
INFO:inspire:Identified number of classes: 2
INFO:inspire:Fitting KNN index for validation set... (0.00 seconds)
INFO:inspire:Identifying border regions... (0.00 seconds)
INFO:inspire:Training models (step 0)... (0.05 seconds)
INFO:inspire:Performing adaptive optimizations (step 1)... (0.00 seconds)
INFO:inspire:Training models (step 1)... (0.06 seconds)
INFO:inspire:Performing adaptive optimizations (step 2)... (0.00 seconds)
INFO:inspire:Training models (step 2)... (0.06 seconds)
INFO:inspire:Performing adaptive optimizations (step 3)... (0.00 seconds)
INFO:inspire:Training models (step 3)... (0.07 seconds)
INFO:inspire:Performing adaptive optimizations (step 4)... (0.00 seconds)
INFO:inspire:Training models (step 4)... (0.07 seconds)
INFO:inspire:Performing adaptive optimizations (step 5)... (0.00 seconds)
INFO:inspire:Training models (step 5)..

InspireClassifier(base_estimator=DecisionTreeClassifier(max_depth=10,
                                                        max_features=0.8,
                                                        min_samples_leaf=3,
                                                        min_samples_split=5,
                                                        random_state=42),
                  n_estimators=100)

In [91]:
generate_training_raport(model, X_test, y_test)

              precision    recall  f1-score   support

           0       0.81      1.00      0.89        25
           1       1.00      0.68      0.81        19

    accuracy                           0.86        44
   macro avg       0.90      0.84      0.85        44
weighted avg       0.89      0.86      0.86        44
 

_________Training report_________

Removed noise: 0
Border samples: 79
Number of folds: 8

Batch 1
Wrong predictions: 4
Low confidence in predictions: 0
Val BP samples: 4
BP samples: 5
Oversampling indces: 5
X_synth size: 75

Batch 2
Changed predictions: 0
Wrong predictions: 4
Low confidence in predictions: 5
Val BP samples: 8
BP samples: 11
Oversampling indces: 11
X_synth size: 75

Batch 3
Changed predictions: 1
Wrong predictions: 5
Low confidence in predictions: 5
Val BP samples: 8
BP samples: 11
Oversampling indces: 11
X_synth size: 75

Batch 4
Changed predictions: 0
Wrong predictions: 5
Low confidence in predictions: 0
Val BP samples: 5
BP samples: 7
Oversamp

In [92]:
recalls = []
precisions = []
fscores = []

for i in tqdm(range(30)):
    model_i = InspireClassifier(
        n_estimators=100,
        base_estimator=DecisionTreeClassifier(
            max_depth=10,
            min_samples_split=5,
            min_samples_leaf=3,
            max_features=0.8,
            random_state=42,
        ),
    )

    model_i.fit(
        X_train,
        y_train,
        X_val,
        y_val,
        save_history_=False,
        oversampling_ratio=1,
        bagging_=False,
        cleanup_=True,
        level=logging.CRITICAL,
        bp_kwargs={"neighbours": 2},
        br_kwargs={"br_treshold": 0.7},
    )

    y_pred = model_i.predict(
        X_test,
        level=logging.CRITICAL,
    )
    report = classification_report(y_test, y_pred, output_dict=True)

    recalls.append(report[f"{minority_class}"]["recall"])
    precisions.append(report[f"{minority_class}"]["precision"])
    fscores.append(report[f"{minority_class}"]["f1-score"])

  0%|          | 0/30 [00:00<?, ?it/s]

In [93]:
ionosphere_results["no_bagging"] = {
    "recalls": recalls,
    "precisions": precisions,
    "fscores": fscores,
}
del model
del model_i

### No SMOTE

In [94]:
model = InspireClassifier(
    n_estimators=100,
    base_estimator=DecisionTreeClassifier(
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    ),
)

model.fit(
    X_train,
    y_train,
    X_val,
    y_val,
    save_history_=True,
    oversampling_ratio=1,
    perform_oversampling_=False,
    cleanup_=False,
    level=logging.INFO,
    bp_kwargs={"neighbours": 2},
    br_kwargs={"br_treshold": 0.7}
)

INFO:inspire:Fitting KNN index... (0.01 seconds)
INFO:inspire:Removing outliers... (0.00 seconds)
INFO:inspire:Identified minority class: 1
INFO:inspire:Identified number of classes: 2
INFO:inspire:Fitting KNN index for validation set... (0.00 seconds)
INFO:inspire:Training models (step 0)... (0.00 seconds)
INFO:inspire:Performing adaptive optimizations (step 1)... (0.00 seconds)
INFO:inspire:Training models (step 1)... (0.00 seconds)
INFO:inspire:Performing adaptive optimizations (step 2)... (0.00 seconds)
INFO:inspire:Training models (step 2)... (0.00 seconds)
INFO:inspire:Performing adaptive optimizations (step 3)... (0.00 seconds)
INFO:inspire:Training models (step 3)... (0.00 seconds)
INFO:inspire:Performing adaptive optimizations (step 4)... (0.00 seconds)
INFO:inspire:Training models (step 4)... (0.00 seconds)
INFO:inspire:Performing adaptive optimizations (step 5)... (0.00 seconds)
INFO:inspire:Training models (step 5)... (0.00 seconds)
INFO:inspire:Performing adaptive optimiza

InspireClassifier(base_estimator=DecisionTreeClassifier(max_depth=10,
                                                        max_features=0.8,
                                                        min_samples_leaf=3,
                                                        min_samples_split=5,
                                                        random_state=42),
                  n_estimators=100)

In [95]:
generate_training_raport(model, X_test, y_test)

              precision    recall  f1-score   support

           0       0.82      0.92      0.87        25
           1       0.88      0.74      0.80        19

    accuracy                           0.84        44
   macro avg       0.85      0.83      0.83        44
weighted avg       0.84      0.84      0.84        44
 

_________Training report_________

Removed noise: 0

Batch 1

Batch 2
Changed predictions: 0

Batch 3
Changed predictions: 0

Batch 4
Changed predictions: 0

Batch 5
Changed predictions: 0

Batch 6
Changed predictions: 0

Batch 7
Changed predictions: 0

Batch 8
Changed predictions: 0

Batch 9
Changed predictions: 0

Batch 10
Changed predictions: 0

Batch 11
Changed predictions: 0

Batch 12
Changed predictions: 0

Batch 13
Changed predictions: 0

Batch 14
Changed predictions: 0

Batch 15
Changed predictions: 0

Batch 16
Changed predictions: 0

Batch 17
Changed predictions: 0

Batch 18
Changed predictions: 0

Batch 19
Changed predictions: 0

Batch 20
Changed predic

In [96]:
recalls = []
precisions = []
fscores = []

for i in tqdm(range(30)):
    model_i = InspireClassifier(
        n_estimators=100,
        base_estimator=DecisionTreeClassifier(
            max_depth=10,
            min_samples_split=5,
            min_samples_leaf=3,
            max_features=0.8,
            random_state=42,
        ),
    )

    model_i.fit(
        X_train,
        y_train,
        X_val,
        y_val,
        save_history_=False,
        oversampling_ratio=1,
        perform_oversampling_=False,
        cleanup_=True,
        level=logging.CRITICAL,
        bp_kwargs={"neighbours": 2},
        br_kwargs={"br_treshold": 0.7},
    )

    y_pred = model_i.predict(
        X_test,
        level=logging.CRITICAL,
    )
    report = classification_report(y_test, y_pred, output_dict=True)

    recalls.append(report[f"{minority_class}"]["recall"])
    precisions.append(report[f"{minority_class}"]["precision"])
    fscores.append(report[f"{minority_class}"]["f1-score"])

  0%|          | 0/30 [00:00<?, ?it/s]

In [97]:
ionosphere_results["no_smote"] = {
    "recalls": recalls,
    "precisions": precisions,
    "fscores": fscores,
}
del model
del model_i

## RFC + BorderlineSMOTE

In [98]:
rfc = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=3,
    max_features=0.8,
    random_state=42,
)

In [99]:
smote = BorderlineSMOTE(kind='borderline-1', random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

In [100]:
rfc.fit(X_resampled, y_resampled)

RandomForestClassifier(max_depth=5, max_features=0.8, min_samples_leaf=3,
                       min_samples_split=5, random_state=42)

In [101]:
y_pred_rfc = rfc.predict(X_test)

print(classification_report(y_test, y_pred_rfc))

              precision    recall  f1-score   support

           0       0.92      0.96      0.94        25
           1       0.94      0.89      0.92        19

    accuracy                           0.93        44
   macro avg       0.93      0.93      0.93        44
weighted avg       0.93      0.93      0.93        44



In [102]:
recalls = []
precisions = []
fscores = []
for i in tqdm(range(30)):
    rfc_i = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    )
    rfc_i.fit(X_resampled, y_resampled)
    
    y_pred = rfc_i.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)

    recalls.append(report[f"{minority_class}"]["recall"])
    precisions.append(report[f"{minority_class}"]["precision"])
    fscores.append(report[f"{minority_class}"]["f1-score"])

  0%|          | 0/30 [00:00<?, ?it/s]

In [103]:
ionosphere_results["rfc_smote"] = {
    "recalls": recalls,
    "precisions": precisions,
    "fscores": fscores,
}

In [104]:
del rfc
del rfc_i

### RFC

In [105]:
rfc = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=3,
    max_features=0.8,
    random_state=42,
)

In [106]:
rfc.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, max_features=0.8, min_samples_leaf=3,
                       min_samples_split=5, random_state=42)

In [107]:
y_pred_rfc = rfc.predict(X_test)

print(classification_report(y_test, y_pred_rfc))

              precision    recall  f1-score   support

           0       0.89      1.00      0.94        25
           1       1.00      0.84      0.91        19

    accuracy                           0.93        44
   macro avg       0.95      0.92      0.93        44
weighted avg       0.94      0.93      0.93        44



In [108]:
recalls = []
precisions = []
fscores = []
for i in tqdm(range(10)):
    rfc_i = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    )
    rfc_i.fit(X_test, y_test)
    
    y_pred = rfc_i.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)

    recalls.append(report[f"{minority_class}"]["recall"])
    precisions.append(report[f"{minority_class}"]["precision"])
    fscores.append(report[f"{minority_class}"]["f1-score"])

  0%|          | 0/10 [00:00<?, ?it/s]

In [109]:
ionosphere_results["rfc"] = {
    "recalls": recalls,
    "precisions": precisions,
    "fscores": fscores,
}

In [110]:
del rfc
del rfc_i

In [111]:
with open("results/ionosphere.json", "w") as f:
    json.dump(ionosphere_results, f, indent=4)

In [112]:
del ionosphere_results

# Pima Indians Diabetes

## Data

In [113]:
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "uciml/pima-indians-diabetes-database",
  "diabetes.csv",
  )

In [114]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [115]:
X = df.drop("Outcome", axis=1).to_numpy()
y = df["Outcome"].to_numpy()

In [116]:
Counter(y)

Counter({np.int64(0): 500, np.int64(1): 268})

In [117]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, shuffle=True
)
X_test, X_val, y_test, y_val = train_test_split(
    X_test, y_test, test_size=0.5, random_state=42, shuffle=True
)

In [118]:
print("train disbalance:", Counter(y_train))
print("test disbalance:", Counter(y_test))
print("val disbalance:", Counter(y_val))

train disbalance: Counter({np.int64(0): 377, np.int64(1): 199})
test disbalance: Counter({np.int64(0): 64, np.int64(1): 32})
val disbalance: Counter({np.int64(0): 59, np.int64(1): 37})


In [119]:
minority_class = 1

## INSPIRE

### All features

In [120]:
model = InspireClassifier(
    n_estimators=100,
    base_estimator=DecisionTreeClassifier(
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    ),
)

model.fit(
    X_train,
    y_train,
    X_val,
    y_val,
    save_history_=True,
    oversampling_ratio=1,
    cleanup_=False,
    level=logging.INFO,
    bp_kwargs={"neighbours": 2},
    br_kwargs={"br_treshold": 0.7}
)

INFO:inspire:Fitting KNN index... (0.03 seconds)
INFO:inspire:Removing outliers... (0.00 seconds)
INFO:inspire:Identified minority class: 1
INFO:inspire:Identified number of classes: 2
INFO:inspire:Fitting KNN index for validation set... (0.00 seconds)
INFO:inspire:Identifying border regions... (0.00 seconds)
INFO:inspire:Training models (step 0)... (0.00 seconds)
INFO:inspire:Performing adaptive optimizations (step 1)... (0.00 seconds)
INFO:inspire:Training models (step 1)... (0.00 seconds)
INFO:inspire:Performing adaptive optimizations (step 2)... (0.00 seconds)
INFO:inspire:Training models (step 2)... (0.00 seconds)
INFO:inspire:Performing adaptive optimizations (step 3)... (0.00 seconds)
INFO:inspire:Training models (step 3)... (0.00 seconds)
INFO:inspire:Performing adaptive optimizations (step 4)... (0.00 seconds)
INFO:inspire:Training models (step 4)... (0.00 seconds)
INFO:inspire:Performing adaptive optimizations (step 5)... (0.00 seconds)
INFO:inspire:Training models (step 5)..

InspireClassifier(base_estimator=DecisionTreeClassifier(max_depth=10,
                                                        max_features=0.8,
                                                        min_samples_leaf=3,
                                                        min_samples_split=5,
                                                        random_state=42),
                  n_estimators=100)

In [121]:
generate_training_raport(model, X_test, y_test)

              precision    recall  f1-score   support

           0       0.86      0.78      0.82        64
           1       0.63      0.75      0.69        32

    accuracy                           0.77        96
   macro avg       0.75      0.77      0.75        96
weighted avg       0.79      0.77      0.78        96
 

_________Training report_________

Removed noise: 0
Border samples: 160
Number of folds: 100

Batch 1
Wrong predictions: 32
Low confidence in predictions: 7
Val BP samples: 37
BP samples: 59
Oversampling indces: 56
X_synth size: 178

Batch 2
Changed predictions: 14
Wrong predictions: 26
Low confidence in predictions: 17
Val BP samples: 39
BP samples: 62
Oversampling indces: 57
X_synth size: 178

Batch 3
Changed predictions: 9
Wrong predictions: 33
Low confidence in predictions: 31
Val BP samples: 46
BP samples: 71
Oversampling indces: 66
X_synth size: 178

Batch 4
Changed predictions: 6
Wrong predictions: 27
Low confidence in predictions: 22
Val BP samples: 40
BP

In [122]:
recalls = []
precisions = []
fscores = []

for i in tqdm(range(30)):
    model_i = InspireClassifier(
        n_estimators=100,
        base_estimator=DecisionTreeClassifier(
            max_depth=10,
            min_samples_split=5,
            min_samples_leaf=3,
            max_features=0.8,
            random_state=42,
        ),
    )

    model_i.fit(
        X_train,
        y_train,
        X_val,
        y_val,
        save_history_=False,
        oversampling_ratio=1,
        cleanup_=True,
        level=logging.CRITICAL,
        bp_kwargs={"neighbours": 2},
        br_kwargs={"br_treshold": 0.7},
    )

    y_pred = model_i.predict(
        X_test,
        level=logging.CRITICAL,
    )
    report = classification_report(y_test, y_pred, output_dict=True)

    recalls.append(report[f"{minority_class}"]["recall"])
    precisions.append(report[f"{minority_class}"]["precision"])
    fscores.append(report[f"{minority_class}"]["f1-score"])

  0%|          | 0/30 [00:00<?, ?it/s]

In [123]:
pima_results = {}
pima_results["all"] = {
    "recalls": recalls,
    "precisions": precisions,
    "fscores": fscores,
}
del model
del model_i

### No bagging

In [124]:
model = InspireClassifier(
    n_estimators=100,
    base_estimator=DecisionTreeClassifier(
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    ),
)

model.fit(
    X_train,
    y_train,
    X_val,
    y_val,
    save_history_=True,
    oversampling_ratio=1,
    bagging_=False,
    cleanup_=False,
    level=logging.INFO,
    bp_kwargs={"neighbours": 2},
    br_kwargs={"br_treshold": 0.7}
)

INFO:inspire:Fitting KNN index... (0.02 seconds)
INFO:inspire:Removing outliers... (0.00 seconds)
INFO:inspire:Identified minority class: 1
INFO:inspire:Identified number of classes: 2
INFO:inspire:Fitting KNN index for validation set... (0.00 seconds)
INFO:inspire:Identifying border regions... (0.00 seconds)
INFO:inspire:Training models (step 0)... (0.09 seconds)
INFO:inspire:Performing adaptive optimizations (step 1)... (0.00 seconds)
INFO:inspire:Training models (step 1)... (0.04 seconds)
INFO:inspire:Performing adaptive optimizations (step 2)... (0.00 seconds)
INFO:inspire:Training models (step 2)... (0.04 seconds)
INFO:inspire:Performing adaptive optimizations (step 3)... (0.00 seconds)
INFO:inspire:Training models (step 3)... (0.04 seconds)
INFO:inspire:Performing adaptive optimizations (step 4)... (0.00 seconds)
INFO:inspire:Training models (step 4)... (0.04 seconds)
INFO:inspire:Performing adaptive optimizations (step 5)... (0.00 seconds)
INFO:inspire:Training models (step 5)..

InspireClassifier(base_estimator=DecisionTreeClassifier(max_depth=10,
                                                        max_features=0.8,
                                                        min_samples_leaf=3,
                                                        min_samples_split=5,
                                                        random_state=42),
                  n_estimators=100)

In [125]:
generate_training_raport(model, X_test, y_test)

              precision    recall  f1-score   support

           0       0.84      0.75      0.79        64
           1       0.59      0.72      0.65        32

    accuracy                           0.74        96
   macro avg       0.72      0.73      0.72        96
weighted avg       0.76      0.74      0.74        96
 

_________Training report_________

Removed noise: 0
Border samples: 160
Number of folds: 8

Batch 1
Wrong predictions: 35
Low confidence in predictions: 0
Val BP samples: 35
BP samples: 59
Oversampling indces: 54
X_synth size: 178

Batch 2
Changed predictions: 8
Wrong predictions: 35
Low confidence in predictions: 20
Val BP samples: 43
BP samples: 71
Oversampling indces: 64
X_synth size: 178

Batch 3
Changed predictions: 10
Wrong predictions: 33
Low confidence in predictions: 30
Val BP samples: 47
BP samples: 73
Oversampling indces: 66
X_synth size: 178

Batch 4
Changed predictions: 6
Wrong predictions: 33
Low confidence in predictions: 10
Val BP samples: 38
BP s

In [126]:
recalls = []
precisions = []
fscores = []

for i in tqdm(range(30)):
    model_i = InspireClassifier(
        n_estimators=100,
        base_estimator=DecisionTreeClassifier(
            max_depth=10,
            min_samples_split=5,
            min_samples_leaf=3,
            max_features=0.8,
            random_state=42,
        ),
    )

    model_i.fit(
        X_train,
        y_train,
        X_val,
        y_val,
        save_history_=False,
        oversampling_ratio=1,
        bagging_=False,
        cleanup_=True,
        level=logging.CRITICAL,
        bp_kwargs={"neighbours": 2},
        br_kwargs={"br_treshold": 0.7},
    )

    y_pred = model_i.predict(
        X_test,
        level=logging.CRITICAL,
    )
    report = classification_report(y_test, y_pred, output_dict=True)

    recalls.append(report[f"{minority_class}"]["recall"])
    precisions.append(report[f"{minority_class}"]["precision"])
    fscores.append(report[f"{minority_class}"]["f1-score"])

  0%|          | 0/30 [00:00<?, ?it/s]

In [127]:
pima_results["no_bagging"] = {
    "recalls": recalls,
    "precisions": precisions,
    "fscores": fscores,
}
del model
del model_i

### No SMOTE

In [128]:
model = InspireClassifier(
    n_estimators=100,
    base_estimator=DecisionTreeClassifier(
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    ),
)

model.fit(
    X_train,
    y_train,
    X_val,
    y_val,
    save_history_=True,
    oversampling_ratio=1,
    perform_oversampling_=False,
    cleanup_=False,
    level=logging.INFO,
    bp_kwargs={"neighbours": 2},
    br_kwargs={"br_treshold": 0.7}
)

INFO:inspire:Fitting KNN index... (0.03 seconds)
INFO:inspire:Removing outliers... (0.00 seconds)
INFO:inspire:Identified minority class: 1
INFO:inspire:Identified number of classes: 2
INFO:inspire:Fitting KNN index for validation set... (0.00 seconds)
INFO:inspire:Training models (step 0)... (0.00 seconds)
INFO:inspire:Performing adaptive optimizations (step 1)... (0.00 seconds)
INFO:inspire:Training models (step 1)... (0.00 seconds)
INFO:inspire:Performing adaptive optimizations (step 2)... (0.00 seconds)
INFO:inspire:Training models (step 2)... (0.00 seconds)
INFO:inspire:Performing adaptive optimizations (step 3)... (0.00 seconds)
INFO:inspire:Training models (step 3)... (0.00 seconds)
INFO:inspire:Performing adaptive optimizations (step 4)... (0.00 seconds)
INFO:inspire:Training models (step 4)... (0.00 seconds)
INFO:inspire:Performing adaptive optimizations (step 5)... (0.00 seconds)
INFO:inspire:Training models (step 5)... (0.00 seconds)
INFO:inspire:Performing adaptive optimiza

InspireClassifier(base_estimator=DecisionTreeClassifier(max_depth=10,
                                                        max_features=0.8,
                                                        min_samples_leaf=3,
                                                        min_samples_split=5,
                                                        random_state=42),
                  n_estimators=100)

In [129]:
generate_training_raport(model, X_test, y_test)

              precision    recall  f1-score   support

           0       0.82      0.80      0.81        64
           1       0.62      0.66      0.64        32

    accuracy                           0.75        96
   macro avg       0.72      0.73      0.72        96
weighted avg       0.75      0.75      0.75        96
 

_________Training report_________

Removed noise: 0

Batch 1

Batch 2
Changed predictions: 4

Batch 3
Changed predictions: 2

Batch 4
Changed predictions: 5

Batch 5
Changed predictions: 3

Batch 6
Changed predictions: 1

Batch 7
Changed predictions: 1

Batch 8
Changed predictions: 0

Batch 9
Changed predictions: 3

Batch 10
Changed predictions: 0

Batch 11
Changed predictions: 1

Batch 12
Changed predictions: 2

Batch 13
Changed predictions: 1

Batch 14
Changed predictions: 1

Batch 15
Changed predictions: 1

Batch 16
Changed predictions: 0

Batch 17
Changed predictions: 0

Batch 18
Changed predictions: 0

Batch 19
Changed predictions: 0

Batch 20
Changed predic

In [130]:
recalls = []
precisions = []
fscores = []

for i in tqdm(range(30)):
    model_i = InspireClassifier(
        n_estimators=100,
        base_estimator=DecisionTreeClassifier(
            max_depth=10,
            min_samples_split=5,
            min_samples_leaf=3,
            max_features=0.8,
            random_state=42,
        ),
    )

    model_i.fit(
        X_train,
        y_train,
        X_val,
        y_val,
        save_history_=False,
        oversampling_ratio=1,
        perform_oversampling_=False,
        cleanup_=True,
        level=logging.CRITICAL,
        bp_kwargs={"neighbours": 2},
        br_kwargs={"br_treshold": 0.7},
    )

    y_pred = model_i.predict(
        X_test,
        level=logging.CRITICAL,
    )
    report = classification_report(y_test, y_pred, output_dict=True)

    recalls.append(report[f"{minority_class}"]["recall"])
    precisions.append(report[f"{minority_class}"]["precision"])
    fscores.append(report[f"{minority_class}"]["f1-score"])

  0%|          | 0/30 [00:00<?, ?it/s]

In [131]:
pima_results["no_smote"] = {
    "recalls": recalls,
    "precisions": precisions,
    "fscores": fscores,
}
del model
del model_i

## RFC + BorderlineSMOTE

In [132]:
rfc = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=3,
    max_features=0.8,
    random_state=42,
)

In [133]:
smote = BorderlineSMOTE(kind='borderline-1', random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

In [134]:
rfc.fit(X_resampled, y_resampled)

RandomForestClassifier(max_depth=5, max_features=0.8, min_samples_leaf=3,
                       min_samples_split=5, random_state=42)

In [135]:
y_pred_rfc = rfc.predict(X_test)

print(classification_report(y_test, y_pred_rfc))

              precision    recall  f1-score   support

           0       0.85      0.72      0.78        64
           1       0.57      0.75      0.65        32

    accuracy                           0.73        96
   macro avg       0.71      0.73      0.71        96
weighted avg       0.76      0.73      0.74        96



In [136]:
recalls = []
precisions = []
fscores = []
for i in tqdm(range(30)):
    rfc_i = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    )
    rfc_i.fit(X_resampled, y_resampled)
    
    y_pred = rfc_i.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)

    recalls.append(report[f"{minority_class}"]["recall"])
    precisions.append(report[f"{minority_class}"]["precision"])
    fscores.append(report[f"{minority_class}"]["f1-score"])

  0%|          | 0/30 [00:00<?, ?it/s]

In [137]:
pima_results["rfc_smote"] = {
    "recalls": recalls,
    "precisions": precisions,
    "fscores": fscores,
}

In [138]:
del rfc
del rfc_i

### RFC

In [139]:
rfc = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=3,
    max_features=0.8,
    random_state=42,
)

In [140]:
rfc.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, max_features=0.8, min_samples_leaf=3,
                       min_samples_split=5, random_state=42)

In [141]:
y_pred_rfc = rfc.predict(X_test)

print(classification_report(y_test, y_pred_rfc))

              precision    recall  f1-score   support

           0       0.83      0.81      0.82        64
           1       0.64      0.66      0.65        32

    accuracy                           0.76        96
   macro avg       0.73      0.73      0.73        96
weighted avg       0.76      0.76      0.76        96



In [142]:
recalls = []
precisions = []
fscores = []
for i in tqdm(range(10)):
    rfc_i = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    )
    rfc_i.fit(X_test, y_test)
    
    y_pred = rfc_i.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)

    recalls.append(report[f"{minority_class}"]["recall"])
    precisions.append(report[f"{minority_class}"]["precision"])
    fscores.append(report[f"{minority_class}"]["f1-score"])

  0%|          | 0/10 [00:00<?, ?it/s]

In [143]:
pima_results["rfc"] = {
    "recalls": recalls,
    "precisions": precisions,
    "fscores": fscores,
}

In [144]:
del rfc
del rfc_i

In [145]:
with open("results/pima_results.json", "w") as f:
    json.dump(pima_results, f, indent=4)

In [146]:
del pima_results

# Credit card fraud 

## Data

In [147]:
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "mlg-ulb/creditcardfraud",
  "creditcard.csv",
)

100%|██████████| 66.0M/66.0M [00:04<00:00, 14.5MB/s]

Extracting zip of creditcard.csv...


In [148]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     28

In [153]:
X = df.drop("Class", axis=1).to_numpy()
y = df["Class"].to_numpy()

In [154]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, shuffle=True
)
X_test, X_val, y_test, y_val = train_test_split(
    X_test, y_test, test_size=0.5, random_state=42, shuffle=True
)

In [155]:
print("train disbalance:", Counter(y_train))
print("test disbalance:", Counter(y_test))
print("val disbalance:", Counter(y_val))

train disbalance: Counter({np.int64(0): 213226, np.int64(1): 379})
test disbalance: Counter({np.int64(0): 35544, np.int64(1): 57})
val disbalance: Counter({np.int64(0): 35545, np.int64(1): 56})


In [156]:
minority_class = 1

## INSPIRE

### All features

In [158]:
model = InspireClassifier(
    n_estimators=100,
    base_estimator=DecisionTreeClassifier(
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    ),
)

model.fit(
    X_train,
    y_train,
    X_val,
    y_val,
    save_history_=True,
    oversampling_ratio=0.1,
    cleanup_=False,
    level=logging.INFO,
    bp_kwargs={"neighbours": 2},
    br_kwargs={"br_treshold": 0.7}
)

INFO:inspire:Fitting KNN index... (8.76 seconds)
INFO:inspire:Removing outliers... (0.03 seconds)
INFO:inspire:Identified minority class: 1
INFO:inspire:Identified number of classes: 2
INFO:inspire:Fitting KNN index for validation set... (0.27 seconds)
INFO:inspire:Identifying border regions... (0.22 seconds)
INFO:inspire:Training models (step 0)... (6.17 seconds)
INFO:inspire:Performing adaptive optimizations (step 1)... (0.10 seconds)
INFO:inspire:Training models (step 1)... (6.48 seconds)
INFO:inspire:Performing adaptive optimizations (step 2)... (0.10 seconds)
INFO:inspire:Training models (step 2)... (6.53 seconds)
INFO:inspire:Performing adaptive optimizations (step 3)... (0.10 seconds)
INFO:inspire:Training models (step 3)... (6.42 seconds)
INFO:inspire:Performing adaptive optimizations (step 4)... (0.10 seconds)
INFO:inspire:Training models (step 4)... (6.45 seconds)
INFO:inspire:Performing adaptive optimizations (step 5)... (0.11 seconds)
INFO:inspire:Training models (step 5)..

InspireClassifier(base_estimator=DecisionTreeClassifier(max_depth=10,
                                                        max_features=0.8,
                                                        min_samples_leaf=3,
                                                        min_samples_split=5,
                                                        random_state=42),
                  n_estimators=100)

In [159]:
generate_training_raport(model, X_test, y_test)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     35544
           1       0.64      0.86      0.73        57

    accuracy                           1.00     35601
   macro avg       0.82      0.93      0.87     35601
weighted avg       1.00      1.00      1.00     35601
 

_________Training report_________

Removed noise: 0
Border samples: 379
Number of folds: 100

Batch 1
Wrong predictions: 24
Low confidence in predictions: 11
Val BP samples: 28
BP samples: 52
Oversampling indces: 52
X_synth size: 21284

Batch 2
Changed predictions: 14
Wrong predictions: 18
Low confidence in predictions: 60
Val BP samples: 74
BP samples: 108
Oversampling indces: 108
X_synth size: 21284

Batch 3
Changed predictions: 25
Wrong predictions: 35
Low confidence in predictions: 104
Val BP samples: 116
BP samples: 150
Oversampling indces: 150
X_synth size: 21284

Batch 4
Changed predictions: 12
Wrong predictions: 33
Low confidence in predictions: 44
Val BP

In [ ]:
model = InspireClassifier(
    n_estimators=100,
    base_estimator=DecisionTreeClassifier(
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    ),
)

model.fit(
    X_train,
    y_train,
    X_val,
    y_val,
    save_history_=True,
    oversampling_ratio=0.5,
    cleanup_=False,
    level=logging.INFO,
    bp_kwargs={"neighbours": 2},
    br_kwargs={"br_treshold": 0.7},
)

In [162]:
generate_training_raport(model, X_test, y_test)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     35544
           1       0.77      0.86      0.81        57

    accuracy                           1.00     35601
   macro avg       0.88      0.93      0.90     35601
weighted avg       1.00      1.00      1.00     35601
 

_________Training report_________

Removed noise: 0
Border samples: 379
Number of folds: 8

Batch 1
Wrong predictions: 20
Low confidence in predictions: 0
Val BP samples: 20
BP samples: 38
Oversampling indces: 38
X_synth size: 21284

Batch 2
Changed predictions: 3
Wrong predictions: 17
Low confidence in predictions: 33
Val BP samples: 43
BP samples: 76
Oversampling indces: 76
X_synth size: 21284

Batch 3
Changed predictions: 13
Wrong predictions: 20
Low confidence in predictions: 48
Val BP samples: 58
BP samples: 94
Oversampling indces: 94
X_synth size: 21284

Batch 4
Changed predictions: 1
Wrong predictions: 19
Low confidence in predictions: 10
Val BP samples: 2

## No bagging

In [160]:
model = InspireClassifier(
    n_estimators=10,
    base_estimator=DecisionTreeClassifier(
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    ),
)

model.fit(
    X_train,
    y_train,
    X_val,
    y_val,
    save_history_=True,
    oversampling_ratio=0.1,
    bagging_=False,
    cleanup_=False,
    level=logging.INFO,
    bp_kwargs={"neighbours": 2},
    br_kwargs={"br_treshold": 0.7},
)

INFO:inspire:Fitting KNN index... (10.07 seconds)
INFO:inspire:Removing outliers... (0.03 seconds)
INFO:inspire:Identified minority class: 1
INFO:inspire:Identified number of classes: 2
INFO:inspire:Fitting KNN index for validation set... (0.29 seconds)
INFO:inspire:Identifying border regions... (0.23 seconds)
INFO:inspire:Training models (step 0)... (32.58 seconds)
INFO:inspire:Performing adaptive optimizations (step 1)... (0.12 seconds)
INFO:inspire:Training models (step 1)... (34.07 seconds)
INFO:inspire:Performing adaptive optimizations (step 2)... (0.11 seconds)
INFO:inspire:Training models (step 2)... (33.12 seconds)
INFO:inspire:Performing adaptive optimizations (step 3)... (0.12 seconds)
INFO:inspire:Training models (step 3)... (32.60 seconds)
INFO:inspire:Performing adaptive optimizations (step 4)... (0.10 seconds)
INFO:inspire:Training models (step 4)... (29.71 seconds)


InspireClassifier(base_estimator=DecisionTreeClassifier(max_depth=10,
                                                        max_features=0.8,
                                                        min_samples_leaf=3,
                                                        min_samples_split=5,
                                                        random_state=42))

In [161]:
generate_training_raport(model, X_test, y_test)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     35544
           1       0.77      0.86      0.81        57

    accuracy                           1.00     35601
   macro avg       0.88      0.93      0.90     35601
weighted avg       1.00      1.00      1.00     35601
 

_________Training report_________

Removed noise: 0
Border samples: 379
Number of folds: 8

Batch 1
Wrong predictions: 20
Low confidence in predictions: 0
Val BP samples: 20
BP samples: 38
Oversampling indces: 38
X_synth size: 21284

Batch 2
Changed predictions: 3
Wrong predictions: 17
Low confidence in predictions: 33
Val BP samples: 43
BP samples: 76
Oversampling indces: 76
X_synth size: 21284

Batch 3
Changed predictions: 13
Wrong predictions: 20
Low confidence in predictions: 48
Val BP samples: 58
BP samples: 94
Oversampling indces: 94
X_synth size: 21284

Batch 4
Changed predictions: 1
Wrong predictions: 19
Low confidence in predictions: 10
Val BP samples: 2

## RFC + BorderlineSMOTE

In [163]:
rfc = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=3,
    max_features=0.8,
    random_state=42,
)

In [164]:
smote = BorderlineSMOTE(kind='borderline-1', random_state=42, sampling_strategy=0.5)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

In [165]:
rfc.fit(X_resampled, y_resampled)

RandomForestClassifier(max_depth=5, max_features=0.8, min_samples_leaf=3,
                       min_samples_split=5, random_state=42)

In [166]:
y_pred_rfc = rfc.predict(X_test)

print(classification_report(y_test, y_pred_rfc))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     35544
           1       0.57      0.86      0.69        57

    accuracy                           1.00     35601
   macro avg       0.78      0.93      0.84     35601
weighted avg       1.00      1.00      1.00     35601

